# OML-47 Production Data Analysis

This notebook walks through the end-to-end data pipeline for a simulated Nigerian oil field (OML-47 — Eko Field).

**Pipeline stages:**
1. Generate realistic daily production data (2 years)
2. Run DuckDB aggregation queries
3. Visualize key production KPIs
4. Export analysis-ready datasets

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## Step 1: Generate Production Data

We simulate 730 days of daily production data for the Eko Field, including:
- Crude oil production (bbl/day) with natural decline curve
- Associated gas production (MMscf/day)
- Water cut percentage (increasing over time)
- OPEX per barrel
- Downtime events

In [ ]:
# Generate simulated data inline (mirrors generate_data.py)
np.random.seed(42)

start_date = datetime(2023, 1, 1)
days = 730
dates = [start_date + timedelta(days=i) for i in range(days)]

# Base production with decline curve (exponential decay)
decline_rate = 0.0003  # Daily decline rate
base_production = 45000  # Initial bbl/day

records = []
for i, date in enumerate(dates):
    # Natural decline + random noise
    oil_prod = base_production * np.exp(-decline_rate * i) + np.random.normal(0, 800)
    
    # Downtime events (5% chance)
    is_downtime = np.random.random() < 0.05
    if is_downtime:
        oil_prod *= np.random.uniform(0.1, 0.5)
    
    # Gas production (GOR ~1.2 Mscf/bbl)
    gas_prod = oil_prod * 1.2 / 1000 + np.random.normal(0, 2)
    
    # Water cut increases over time
    water_cut = min(5 + (i / days) * 25 + np.random.normal(0, 2), 40)
    
    # OPEX per barrel (increases with water cut)
    opex = 12 + water_cut * 0.3 + np.random.normal(0, 1.5)
    
    records.append({
        'date': date.strftime('%Y-%m-%d'),
        'oil_bbl': max(0, round(oil_prod)),
        'gas_mmscf': max(0, round(gas_prod, 2)),
        'water_cut_pct': max(0, round(water_cut, 1)),
        'opex_per_bbl': max(5, round(opex, 2)),
        'downtime': is_downtime
    })

df = pd.DataFrame(records)
df['date'] = pd.to_datetime(df['date'])
print(f'Generated {len(df)} days of production data')
print(f'Date range: {df.date.min().strftime("%Y-%m-%d")} to {df.date.max().strftime("%Y-%m-%d")}')
df.head(10)

## Step 2: DuckDB Aggregation Pipeline

We use DuckDB for fast in-process SQL analytics — computing monthly production summaries,
lifting cost per barrel, and revenue scenarios at different Brent prices.

In [ ]:
con = duckdb.connect()
con.register('production', df)

# Monthly production summary
monthly = con.execute("""
    SELECT 
        DATE_TRUNC('month', date) AS month,
        SUM(oil_bbl) AS total_oil_bbl,
        SUM(gas_mmscf) AS total_gas_mmscf,
        AVG(water_cut_pct) AS avg_water_cut,
        AVG(opex_per_bbl) AS avg_opex_per_bbl,
        SUM(CASE WHEN downtime THEN 1 ELSE 0 END) AS downtime_days,
        COUNT(*) AS production_days
    FROM production
    GROUP BY DATE_TRUNC('month', date)
    ORDER BY month
""").fetchdf()

# Revenue scenarios
for price in [70, 80, 90]:
    monthly[f'revenue_{price}'] = monthly['total_oil_bbl'] * price

print(f'Monthly summary: {len(monthly)} months')
monthly.head()

## Chart 1: Production Decline Curve

The exponential decline in daily oil production is a key indicator for field planning.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# 7-day rolling average for smoother visualization
df['oil_rolling'] = df['oil_bbl'].rolling(7).mean()

ax.fill_between(df['date'], df['oil_rolling'], alpha=0.3, color='#2ecc71')
ax.plot(df['date'], df['oil_rolling'], color='#2ecc71', linewidth=1.5, label='7-Day Rolling Avg')
ax.scatter(df[df['downtime']]['date'], df[df['downtime']]['oil_bbl'], 
          color='#e74c3c', s=20, alpha=0.7, label='Downtime Events', zorder=5)

ax.set_title('OML-47 Eko Field — Daily Oil Production Decline Curve', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Oil Production (bbl/day)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Chart 2: Cost Per Barrel Trend

OPEX per barrel increases as water cut rises — a critical metric for Seplat-type operations.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# OPEX per barrel
ax1.plot(monthly['month'], monthly['avg_opex_per_bbl'], 
         color='#e67e22', linewidth=2, marker='o', markersize=6)
ax1.fill_between(monthly['month'], monthly['avg_opex_per_bbl'], alpha=0.2, color='#e67e22')
ax1.set_title('Monthly Average OPEX per Barrel', fontsize=14, fontweight='bold')
ax1.set_ylabel('USD/bbl')
ax1.axhline(y=20, color='#e74c3c', linestyle='--', alpha=0.7, label='$20 threshold')
ax1.legend()

# Water cut
ax2.plot(monthly['month'], monthly['avg_water_cut'], 
         color='#3498db', linewidth=2, marker='s', markersize=6)
ax2.fill_between(monthly['month'], monthly['avg_water_cut'], alpha=0.2, color='#3498db')
ax2.set_title('Monthly Average Water Cut', fontsize=14, fontweight='bold')
ax2.set_ylabel('Water Cut (%)')
ax2.set_xlabel('Month')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Chart 3: Revenue Scenario Comparison

Monthly revenue projections at $70, $80, and $90 per barrel Brent crude prices.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'revenue_70': ('#e74c3c', '$70/bbl'), 
          'revenue_80': ('#f39c12', '$80/bbl'), 
          'revenue_90': ('#2ecc71', '$90/bbl')}

for col, (color, label) in colors.items():
    ax.plot(monthly['month'], monthly[col] / 1e6, 
            color=color, linewidth=2, marker='o', markersize=5, label=label)

ax.set_title('Monthly Revenue Scenarios — OML-47 Eko Field', fontsize=16, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (USD Millions)')
ax.legend(title='Brent Price', fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 3: Export to Excel

Export the monthly summary as an analysis-ready Excel file for Power BI integration.

In [ ]:
# Export monthly summary
output_path = 'output/field_summary.xlsx'
monthly.to_excel(output_path, index=False, sheet_name='Monthly Summary')
print(f'Exported to {output_path}')

# Also export raw data as CSV for Power BI
df.to_csv('data/processed/daily_production.csv', index=False)
monthly.to_csv('data/processed/monthly_summary.csv', index=False)
print('CSV exports complete')

## Key Findings

- **Production decline**: ~20% decline over 2 years from natural reservoir depletion
- **Water cut**: Increased from ~5% to ~30%, driving OPEX higher
- **OPEX trend**: Rose from ~$13/bbl to ~$19/bbl, approaching the $20 economic threshold
- **Revenue impact**: At $80/bbl Brent, monthly revenue declined from ~$110M to ~$90M
- **Downtime**: ~5% of days experienced production disruptions

### DuckDB Query Structure

All aggregations use DuckDB's in-process SQL engine for fast analytics without a database server.
Key queries: `DATE_TRUNC` for monthly grouping, `CASE WHEN` for conditional aggregation,
and window functions for rolling calculations.

---
*Pipeline built by Jimi Aboderin | [GitHub](https://github.com/JimiR3d)*